# Joint outcomes in the paper's binary `causl` setting

This is a **separate joint-outcome diagnostic**, not a comparison with Frugal Flows.
We extend the paper's 5-instrument/5-confounder binary-treatment setup to two
continuous outcomes. The R package `causl` generates the observed data with a
Gaussian pair copula. The package's joint model uses `noise_dim=2`; when sampling the causal margin,
`StoNet` also supplies a two-dimensional reference `eta`. A separate randomized-treatment diagnostic checks
whether the specified moments agree with the generated model; it is not used
to define the causal truth. Under either intervention `do(A=a)` the specified causal
margin is bivariate normal with exact mean `(2a, -a)`, unit marginal variances,
and cross-covariance `0.4`. The first outcome retains the paper's five
confounder copula links. The R generator and its settings are in
`examples/causl_joint_outcomes.R`. Ground truth is given by the model parameters,
not by reference Monte Carlo samples.

**Run All** generates one `n=5000` dataset, fits a single joint Frengression
model for 1000 updates, then prints both arms' marginal and dependence errors and compares them
with a longer fit. A failed pilot is not a paper result.
This requires R with `causl` and the Python environment used for the existing
Frengression notebook (`torch`, `engression`, `numpy`, `pandas`, `matplotlib`).
Install R's package with `remotes::install_github('rje42/causl')` if needed.
The notebook contains no saved results and writes nothing into this repository.


In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'src' / 'frengression').is_dir()
             and (p / 'examples' / 'causl_joint_outcomes.R').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the frengression_pkg repository or examples/ folder.')
sys.path.insert(0, str(ROOT / 'src'))
from frengression import Frengression

RSCRIPT = shutil.which('Rscript')
if RSCRIPT is None:
    raise RuntimeError('Rscript was not found. Install R and the R package causl.')
check = subprocess.run([RSCRIPT, '-e', 'stopifnot(requireNamespace("causl", quietly=TRUE))'],
                       capture_output=True, text=True)
if check.returncode:
    raise RuntimeError('R package causl is missing. In R: remotes::install_github("rje42/causl")'
                       + '\n' + check.stderr[-1000:])
print('Repository:', ROOT)
print('Rscript:', RSCRIPT)
print('torch:', torch.__version__)


## Generate the common observational dataset

The causal truth below is **specified by the model parameters**. The
observational residual check cannot by itself verify the interventional law;
the following randomized diagnostic checks the specification separately. `I1`–`I5` influence treatment, while all
five `X` covariates are associated with treatment and `Y1`.


In [ ]:
COVARIATES = [f'I{i}' for i in range(1, 6)] + [f'X{i}' for i in range(1, 6)]
TRUE_COV = np.array([[1.0, 0.4], [0.4, 1.0]])
TRUE_MEAN = {0: np.array([0.0, 0.0]), 1: np.array([2.0, -1.0])}

def generate_causl_joint(n=5000, seed=2026, strength_instr=1.0, randomized=False):
    with tempfile.TemporaryDirectory(prefix='causl_joint_') as temp:
        csv_path = Path(temp) / 'observed.csv'
        cmd = [RSCRIPT, str(ROOT / 'examples' / 'causl_joint_outcomes.R'),
               str(csv_path), str(n), str(seed), str(strength_instr)]
        if randomized:
            cmd.append('randomized')
        run = subprocess.run(cmd, capture_output=True, text=True)
        if run.returncode:
            raise RuntimeError('Native causl generation failed:\n' +
                               run.stderr[-4000:] + '\n' + run.stdout[-1000:])
        frame = pd.read_csv(csv_path)
    expected = COVARIATES + ['A', 'Y1', 'Y2']
    if frame.columns.tolist() != expected or len(frame) != n:
        raise ValueError(f'Unexpected causl columns/rows: {frame.shape}, {frame.columns.tolist()}')
    x = frame[['A']].to_numpy(dtype=np.float32)
    z = frame[COVARIATES].to_numpy(dtype=np.float32)
    y = frame[['Y1', 'Y2']].to_numpy(dtype=np.float32)
    if not np.isfinite(np.concatenate([z, x, y], axis=1)).all() or not np.isin(x, [0, 1]).all():
        raise ValueError('causl produced non-finite values or treatment outside {0,1}')
    if min((x[:, 0] == 0).sum(), (x[:, 0] == 1).sum()) == 0:
        raise ValueError('One observed treatment arm is empty')
    # This sanity check uses the observational residuals, never a simulated truth set.
    residual = y - np.column_stack((2 * x[:, 0], -x[:, 0]))
    empirical_cov = np.cov(residual, rowvar=False)
    if n >= 3000 and abs(empirical_cov[0, 1] - TRUE_COV[0, 1]) > 0.15:
        raise RuntimeError('causl residual covariance does not match the specified 0.4. '
                           f'Observed={empirical_cov[0,1]:.3f}; inspect the R/copula version before fitting.')
    return x, z, y, empirical_cov

N, SEED, STRENGTH_INSTR = 5000, 2026, 1.0
x, z, y, observed_residual_cov = generate_causl_joint(N, SEED, STRENGTH_INSTR)
print('x, z, y shapes:', x.shape, z.shape, y.shape)
print('Observed residual cross-covariance (sanity check):', round(observed_residual_cov[0,1], 3))
print('Specified causal cross-covariance:', TRUE_COV[0,1])


### Check the causal specification independently

The next cell uses the **same `causl` parameters** with randomized treatment
(propensity 1/2) in a separate, larger diagnostic sample. In that sample, each
observed arm approximates the corresponding interventional law. This checks
our interpretation of the two-outcome `causl` copula; its sample moments are
**never used as ground truth** in the fitted-model scores. If they disagree
substantially with the specified mean and covariance, stop and inspect the R
model before fitting.


In [ ]:
x_check, _, y_check, _ = generate_causl_joint(
    n=20000, seed=SEED + 1, strength_instr=STRENGTH_INSTR, randomized=True)
check_rows = []
for arm in (0, 1):
    d = y_check[x_check[:, 0] == arm]
    c = np.cov(d, rowvar=False)
    check_rows.append(dict(arm=arm, count=len(d),
                           mean1=d[:, 0].mean(), expected_mean1=TRUE_MEAN[arm][0],
                           mean2=d[:, 1].mean(), expected_mean2=TRUE_MEAN[arm][1],
                           var1=c[0, 0], var2=c[1, 1],
                           cov12=c[0, 1], expected_cov12=TRUE_COV[0, 1]))
check_table = pd.DataFrame(check_rows)
display(check_table.round(3))
print('Diagnostic only: the exact values in the expected columns define the truth.')


## Fit one joint model

`y_dim=2` and `noise_dim=2` use the package's original joint architecture
(3 layers, width 100, learning rate `1e-4`). `model_y` receives a
2-dimensional reference `eta` and also draws its own 2-dimensional layer
noise, as it does in the scalar experiment. The `noise_dim` parameter is 2.
Crucially, `StoNet.sample` returns `(batch, outcome_dimension, draws)`;
we transpose its axes before calculating any joint moments. The displayed seconds cover only fitting the causal-margin modules.
For a stable paper result, inspect the pilot before setting `RUN_REPEATS=True`.


In [ ]:
FR_ITERS = 1000
MC_DRAWS = 5000
torch.set_num_threads(1)

def fit_joint(x, z, y, seed, iterations=FR_ITERS, draws=MC_DRAWS):
    torch.manual_seed(seed)
    model = Frengression(x_dim=1, y_dim=2, z_dim=10,
                         num_layer=3, hidden_dim=100, noise_dim=2,
                         x_binary=True, z_binary_dims=0, device=torch.device('cpu'))
    xt, zt, yt = (torch.as_tensor(v, dtype=torch.float32) for v in (x, z, y))
    start = time.perf_counter()
    model.train_y(xt, zt, yt, num_iters=iterations, lr=1e-4,
                  print_every_iter=max(1, iterations // 5), tol=0.0)
    seconds = time.perf_counter() - start
    arm_draws = {}
    for arm in (0, 1):
        sample = model.sample_causal_margin(torch.tensor([arm], dtype=torch.int32),
                                            sample_size=draws)
        # StoNet.sample returns [data_size, response_dim, sample_size].
        # Direct reshape(draws, 2) would scramble the two outcomes.
        if tuple(sample.shape) != (1, y.shape[1], draws):
            raise RuntimeError(f'Unexpected joint sample shape: {tuple(sample.shape)}')
        arm_draws[arm] = sample[0].transpose(0, 1).contiguous().cpu().numpy()
        if not np.isfinite(arm_draws[arm]).all():
            raise ValueError(f'Non-finite Frengression sample for arm {arm}')
    return model, arm_draws, seconds

joint_model, joint_draws, fit_seconds = fit_joint(x, z, y, SEED)
print(f'One joint fit: {fit_seconds:.1f} seconds')
print('Frengression noise_dim=2; joint samples transposed to [draws, outcomes]')


## Compare with the exact specified law

Report marginal means and variances **and** the off-diagonal covariance for
each intervention. The shuffled second coordinate below is a diagnostic:
shuffling keeps both fitted one-dimensional margins exactly the same while
removing their dependence. It is not a fitted competing method and must not
be called a Frugal Flows comparison.


In [ ]:
def evaluate_joint(draws, label):
    rows = []
    for arm in (0, 1):
        d = np.asarray(draws[arm], dtype=float)
        mu, cov = d.mean(axis=0), np.cov(d, rowvar=False)
        rows.append(dict(method=label, arm=arm, true_mean1=TRUE_MEAN[arm][0],
                         estimated_mean1=mu[0], true_mean2=TRUE_MEAN[arm][1],
                         estimated_mean2=mu[1], mean_mae=np.abs(mu - TRUE_MEAN[arm]).mean(),
                         true_var1=1.0, estimated_var1=cov[0,0],
                         true_var2=1.0, estimated_var2=cov[1,1],
                         true_cov12=0.4, estimated_cov12=cov[0,1],
                         cov12_error=abs(cov[0,1] - 0.4),
                         covariance_fro_error=np.linalg.norm(cov - TRUE_COV, ord='fro')))
    return pd.DataFrame(rows)

rng = np.random.default_rng(SEED + 99)
shuffled = {arm: np.column_stack((joint_draws[arm][:, 0],
             rng.permutation(joint_draws[arm][:, 1]))) for arm in (0, 1)}
pilot_table = pd.concat([evaluate_joint(joint_draws, 'frengression_joint'),
                         evaluate_joint(shuffled, 'shuffled_diagnostic')], ignore_index=True)
display(pilot_table.round(4))


### Check whether 1000 updates caused the poor joint fit

The 1000-update model is a pilot, not a result ready for the manuscript. Refit
from the **same data and initialization seed** for 4000 updates and compare both
interventions. If either causal mean or the covariance remains far from its
specified value, do not run the repeated experiment or use this example as
evidence of multivariate performance. Investigate optimization or the model
implementation first; do not change the data generator to obtain a better fit.


In [ ]:
LONGER_ITERS = 4000
longer_model, longer_draws, longer_seconds = fit_joint(
    x, z, y, SEED, iterations=LONGER_ITERS)
longer_table = evaluate_joint(longer_draws, f'frengression_{LONGER_ITERS}_updates')
comparison = pd.concat([pilot_table.loc[pilot_table.method == 'frengression_joint'],
                        longer_table], ignore_index=True)
display(comparison[['method', 'arm', 'estimated_mean1', 'estimated_mean2',
                    'mean_mae', 'estimated_var1', 'estimated_var2',
                    'estimated_cov12', 'cov12_error']].round(4))
print(f'Longer fit time: {longer_seconds:.1f} seconds')
if (comparison.loc[comparison.method == f'frengression_{LONGER_ITERS}_updates',
                  'mean_mae'].max() > 0.2 or longer_table.cov12_error.max() > 0.2):
    print('Joint pilot still fails: do not use it in the manuscript or enable replications.')
else:
    print('Pilot improved; check both arms and rerun across seeds before any manuscript claim.')


### Locate the failed joint fit without retraining

The warning about missing covariate dimensions is emitted when `StoNet`
receives only treatment and fills the missing reference-eta coordinates itself.
The check below compares that route with explicitly sampled Gaussian `eta`.
It also checks whether the learned `h(Z,A,xi)` is close to a standard
bivariate normal separately for each treatment, and whether generated
observational outcomes match the observed outcomes within each arm. The
observational arm moments are **not** the interventional truth. These are
triage diagnostics only; they do not validate a causal claim.


In [ ]:
longer_model.eval()
torch.manual_seed(SEED + 501)
rng = np.random.default_rng(SEED + 502)
diagnostic_rows = []
with torch.no_grad():
    xt = torch.as_tensor(x, dtype=torch.float32)
    zt = torch.as_tensor(z, dtype=torch.float32)
    eta_observed = longer_model.model_eta(torch.cat([xt, zt], dim=1))
    y_observed_fit = longer_model.model_y(torch.cat([xt, eta_observed], dim=1)).cpu().numpy()
    for arm in (0, 1):
        mask = x[:, 0] == arm
        x_ref = torch.full((MC_DRAWS, 1), float(arm))
        eta_ref = torch.randn(MC_DRAWS, 2)
        explicit = longer_model.model_y(torch.cat([x_ref, eta_ref], dim=1)).cpu().numpy()
        api = longer_draws[arm]
        # Each z is marginal, independently of the fixed intervention arm.
        z_ref = torch.as_tensor(z[rng.permutation(len(z))], dtype=torch.float32)
        x_latent = torch.full((len(z), 1), float(arm))
        eta_refit = longer_model.model_eta(torch.cat([x_latent, z_ref], dim=1)).cpu().numpy()
        for label, values in [('causal_api', api), ('causal_explicit_eta', explicit),
                              ('eta_given_arm', eta_refit),
                              ('observed_data', y[mask]),
                              ('observed_fit', y_observed_fit[mask])]:
            mu = values.mean(axis=0)
            cov = np.cov(values, rowvar=False)
            diagnostic_rows.append(dict(arm=arm, source=label, n=len(values),
                                        mean1=mu[0], mean2=mu[1],
                                        var1=cov[0, 0], var2=cov[1, 1],
                                        cov12=cov[0, 1]))
diagnostic_table = pd.DataFrame(diagnostic_rows)
display(diagnostic_table.round(3))
print('eta_given_arm target: means (0, 0), variances (1, 1), covariance 0.')
print('causal_api and causal_explicit_eta should agree within simulation error.')
print('observed_data versus observed_fit checks model fit, not causal recovery.')


In [ ]:
# The contour is the analytic 95% ellipse of the specified bivariate normal.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), constrained_layout=True)
grid = np.linspace(0, 2 * np.pi, 250)
eigvals, eigvecs = np.linalg.eigh(TRUE_COV)
unit_circle = np.stack((np.cos(grid), np.sin(grid)))
ell = (eigvecs @ np.diag(np.sqrt(eigvals)) @ unit_circle) * np.sqrt(-2 * np.log(0.05))
for arm, ax in enumerate(axes):
    d = joint_draws[arm]
    subset = np.random.default_rng(SEED + arm).choice(len(d), size=min(700, len(d)), replace=False)
    ax.scatter(d[subset, 0], d[subset, 1], alpha=0.18, s=8, label='Frengression draws')
    contour = ell + TRUE_MEAN[arm][:, None]
    ax.plot(contour[0], contour[1], color='black', lw=2, label='Exact 95% ellipse')
    ax.set(title=f'do(A={arm})', xlabel='Y1', ylabel='Y2')
    ax.set_aspect('equal', adjustable='datalim')
    ax.legend(loc='upper right')
plt.show()


## Optional replications (off by default)

Set `RUN_REPEATS=True` only after both arms pass the preceding
DGP and fit diagnostics. The current repeat block trains for `FR_ITERS=1000` with the package architecture;
change that setting in the fit cell and rerun the pilot first if a longer fit
is needed. Start with five seeds, then extend to 30 after inspecting them. Results are saved under your home
folder (`~/frengression_results`), **outside this Git repository**. The same
analytic truth is used for each seed. No result is included in the notebook.


In [ ]:
RUN_REPEATS = False
if RUN_REPEATS:
    all_rows = []
    for seed in range(2026, 2031):
        xx, zz, yy, _ = generate_causl_joint(N, seed, STRENGTH_INSTR)
        _, draws, seconds = fit_joint(xx, zz, yy, seed)
        table = evaluate_joint(draws, 'frengression_joint')
        table.insert(0, 'seed', seed)
        table['fit_seconds'] = seconds
        all_rows.append(table)
    repeated_results = pd.concat(all_rows, ignore_index=True)
    result_dir = Path.home() / 'frengression_results'
    result_dir.mkdir(exist_ok=True)
    result_path = result_dir / 'joint_outcomes_causl.csv'
    repeated_results.to_csv(result_path, index=False)
    display(repeated_results.groupby('arm')[['mean_mae', 'cov12_error',
                                             'covariance_fro_error', 'fit_seconds']].mean().round(4))
    print('Local results:', result_path)
